# YOLOv8 with MobileNetV2 & GhostNet Backbones — Full and Reduced Parameters

## Wind Turbine Damage Detection — Full Dataset Training

Same data pipeline as `aug-85.ipynb` (class balancing, 70/15/15 split, heavy augmentation).  
This notebook trains **4 custom-backbone models** for direct comparison:

### Models Trained:

| # | Model | Backbone | Channels | Target Params |
|---|-------|----------|----------|--------------|
| 1 | **MobileNetV2 Full** | DWConv + Pointwise | 16→24→48→96→160 | ~1.8M |
| 2 | **MobileNetV2 Reduced** | DWConv + Pointwise | 8→16→32→64→96 | ~0.9M |
| 3 | **GhostNet Full** | GhostConv + C3Ghost | 16→32→64→128→160 | ~1.5M |
| 4 | **GhostNet Reduced** | GhostConv + C3Ghost | 8→16→32→64→96 | ~0.7M |

### Backbone Concepts:

**MobileNetV2:** Depthwise Separable Convolutions (DWConv → Pointwise 1×1)  
→ Splits a standard conv into per-channel spatial + cross-channel mix → ~8x fewer ops

**GhostNet:** Ghost Modules (Primary Conv → Cheap Linear Ghost Maps)  
→ Generates extra feature maps cheaply via depthwise ops on half the filters → ~2x fewer ops vs standard

**Parameter Reduction Strategy:** Halve all channel widths throughout backbone and neck.

**Classes:** damage (0), dirt (1)

In [ ]:
import os
import random
import shutil
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
import cv2
import yaml
import torch

from ultralytics import YOLO

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")

In [ ]:
import json

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR         = Path("/kaggle/working")
PREPROCESSED_DIR = BASE_DIR / "preprocessed_dataset"
BALANCED_DIR     = BASE_DIR / "balanced_dataset"
RUNS_DIR         = BASE_DIR / "runs" / "backbone_comparison"
YAML_DIR         = BASE_DIR / "backbone_yamls"
YAML_DIR.mkdir(parents=True, exist_ok=True)

# ── Raw dataset ───────────────────────────────────────────────────────────────
RAW_DATA_DIR     = Path("/kaggle/input/datasets/ajifoster3/yolo-annotated-wind-turbines-586x371/NordTank586x371")
RAW_IMAGES_DIR   = RAW_DATA_DIR / "images"
RAW_LABELS_DIR   = RAW_DATA_DIR / "labels"
VALID_PAIRS_FILE = RAW_DATA_DIR / "valid_pairs.json"

# ── Class info ────────────────────────────────────────────────────────────────
CLASS_NAMES = ['damage', 'dirt']   # 0 = damage, 1 = dirt
NC = 2

# ── Split ratios ──────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

print("=" * 60)
print("Configuration")
print("=" * 60)
for p, name in [
    (RAW_DATA_DIR,   "Raw dataset root"),
    (RAW_IMAGES_DIR, "Raw images"),
    (RAW_LABELS_DIR, "Raw labels"),
]:
    print(f"  {'✓' if p.exists() else '✗'}  {name}: {p}")

## § 0  Preprocessing — Raw Data → `preprocessed_dataset/`

Identical to `aug-85.ipynb`: resize 586×371 → 640×640, copy labels unchanged.

In [ ]:
PREPROCESSED_IMAGES_DIR = PREPROCESSED_DIR / "images"
PREPROCESSED_LABELS_DIR = PREPROCESSED_DIR / "labels"
PREPROCESSED_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESSED_LABELS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SIZE = (640, 640)

if VALID_PAIRS_FILE.exists():
    with open(VALID_PAIRS_FILE) as f:
        valid_pairs = json.load(f)
    print(f"✓ Loaded {len(valid_pairs)} valid pairs")
else:
    raw_images = set(p.stem for p in RAW_IMAGES_DIR.glob("*.png")) | \
                 set(p.stem for p in RAW_IMAGES_DIR.glob("*.jpg"))
    raw_labels = set(p.stem for p in RAW_LABELS_DIR.glob("*.txt"))
    valid_pairs = list(raw_images & raw_labels)
    print(f"Scanned {len(valid_pairs)} pairs")

processed, skipped = 0, 0
for pair in valid_pairs:
    stem = Path(pair if isinstance(pair, str) else pair.get('image', '')).stem
    img_path = next((RAW_IMAGES_DIR / f"{stem}{e}" for e in ['.png','.jpg','.jpeg']
                     if (RAW_IMAGES_DIR / f"{stem}{e}").exists()), None)
    lbl_path = RAW_LABELS_DIR / f"{stem}.txt"
    if img_path and lbl_path.exists():
        try:
            Image.open(img_path).resize(TARGET_SIZE, Image.Resampling.LANCZOS)\
                 .save(PREPROCESSED_IMAGES_DIR / f"{stem}.png", 'PNG')
            shutil.copy(lbl_path, PREPROCESSED_LABELS_DIR / f"{stem}.txt")
            processed += 1
        except Exception as e:
            skipped += 1
    else:
        skipped += 1

print(f"✓ Preprocessed: {processed} | Skipped: {skipped}")
print(f"  Images: {len(list(PREPROCESSED_IMAGES_DIR.glob('*.png')))}")

## 1. Analyze Dataset & Create Balanced Split

Same strategy as `aug-85.ipynb`: oversample minority class, 70/15/15 split.

In [ ]:
def collect_all_samples(preprocessed_dir):
    imgs_dir   = Path(preprocessed_dir) / "images"
    labels_dir = Path(preprocessed_dir) / "labels"
    all_samples, class_counts, img_per_class = [], Counter(), defaultdict(list)
    for label_file in sorted(labels_dir.glob("*.txt")):
        img_path = next((imgs_dir / f"{label_file.stem}{e}"
                         for e in [".png",".jpg",".jpeg"]
                         if (imgs_dir / f"{label_file.stem}{e}").exists()), None)
        if img_path is None:
            continue
        classes_here = set()
        with open(label_file) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cls = int(parts[0])
                    class_counts[cls] += 1
                    classes_here.add(cls)
        all_samples.append((img_path, label_file))
        for cls in classes_here:
            img_per_class[cls].append(label_file.stem)
    return all_samples, class_counts, img_per_class


def create_balanced_split_dataset(all_samples, img_per_class, output_dir,
                                   train_ratio=0.70, val_ratio=0.15,
                                   class_names=None, seed=42):
    random.seed(seed); np.random.seed(seed)
    output_dir = Path(output_dir)
    if output_dir.exists():
        shutil.rmtree(output_dir)
    for split in ['train','val','test']:
        (output_dir / split / 'images').mkdir(parents=True, exist_ok=True)
        (output_dir / split / 'labels').mkdir(parents=True, exist_ok=True)

    sample_map  = {img.stem: (img, lbl) for img, lbl in all_samples}
    cls_lists   = {cls: list(stems) for cls, stems in img_per_class.items()}
    max_count   = max(len(v) for v in cls_lists.values())

    balanced_stems = []
    for cls, stems in cls_lists.items():
        oversampled = stems[:]
        while len(oversampled) < max_count:
            oversampled.append(random.choice(stems))
        balanced_stems.extend(oversampled)

    seen, final_list = Counter(), []
    for stem in balanced_stems:
        if stem not in sample_map: continue
        img_src, lbl_src = sample_map[stem]
        seen[stem] += 1
        dest_stem = stem if seen[stem] == 1 else f"{stem}_dup{seen[stem]-1}"
        final_list.append((img_src, lbl_src, dest_stem))

    random.shuffle(final_list)
    n = len(final_list)
    n_train = int(n * train_ratio)
    n_val   = int(n * val_ratio)
    splits  = {'train': final_list[:n_train],
                'val':  final_list[n_train:n_train+n_val],
                'test': final_list[n_train+n_val:]}

    for split_name, items in splits.items():
        for img_src, lbl_src, dest_stem in items:
            shutil.copy2(img_src, output_dir/split_name/'images'/f"{dest_stem}{img_src.suffix}")
            shutil.copy2(lbl_src, output_dir/split_name/'labels'/f"{dest_stem}.txt")

    data_yaml = {'path': str(output_dir).replace('\\','/'),
                 'train':'train/images','val':'val/images','test':'test/images',
                 'nc': len(class_names) if class_names else 2,
                 'names': class_names or [f"class_{i}" for i in range(2)]}
    yaml_path = output_dir / 'data.yaml'
    with open(yaml_path,'w') as f:
        yaml.dump(data_yaml, f, default_flow_style=False)

    print("=" * 55)
    print("Balanced Dataset")
    print("=" * 55)
    for s in ['train','val','test']:
        n_img = len(list((output_dir/s/'images').glob('*')))
        print(f"  {s:<6}: {n_img} images")
    return yaml_path


all_samples, raw_counts, raw_img_per_class = collect_all_samples(PREPROCESSED_DIR)

print(f"Total pairs: {len(all_samples)}")
for cls_id, count in sorted(raw_counts.items()):
    name = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else f"cls_{cls_id}"
    print(f"  [{cls_id}] {name}: {count} annotations, {len(raw_img_per_class[cls_id])} images")

In [ ]:
DATA_YAML = create_balanced_split_dataset(
    all_samples   = all_samples,
    img_per_class = raw_img_per_class,
    output_dir    = BALANCED_DIR,
    train_ratio   = TRAIN_RATIO,
    val_ratio     = VAL_RATIO,
    class_names   = CLASS_NAMES,
    seed          = 42,
)
print(f"\ndata.yaml: {DATA_YAML}")

## 2. Define Custom Backbone Architectures

Four YAML files are generated:

| File | Backbone | Channels | Approx Params |
|------|----------|----------|--------------|
| `yolov8_mobilenet_full.yaml` | DWConv + Pointwise | 16→24→48→96→160 | ~1.8M |
| `yolov8_mobilenet_reduced.yaml` | DWConv + Pointwise | 8→16→32→64→96 | ~0.9M |
| `yolov8_ghost_full.yaml` | GhostConv + C3Ghost | 16→32→64→128→160 | ~1.5M |
| `yolov8_ghost_reduced.yaml` | GhostConv + C3Ghost | 8→16→32→64→96 | ~0.7M |

**GhostNet module:**  
Each `GhostConv` generates K/2 primary feature maps via regular conv, then produces K/2 "ghost" maps via cheap depthwise ops on the primaries → same output channels at ~half the cost.

**`C3Ghost`** = C3 bottleneck where inner convolutions use GhostConv.

Neck/Head topology is identical across all four (PANet + YOLOv8 Detect P3/P4/P5).

In [ ]:
# ── 1. MobileNetV2 Full  (16→24→48→96→160) ───────────────────────────────────
MOBILENET_FULL_YAML = YAML_DIR / "yolov8_mobilenet_full.yaml"

mobilenet_full = """\
# MobileNetV2-style backbone (FULL: 16->24->48->96->160 channels) ~1.8M params
nc: 2
scales:
  n: [0.33, 0.25, 1024]

backbone:
  - [-1, 1, Conv,    [16, 3, 2]]          # 0-P1/2
  - [-1, 1, DWConv,  [16, 3, 2]]          # 1-P2/4  depthwise
  - [-1, 1, Conv,    [24, 1, 1]]          # 2        pointwise expand
  - [-1, 1, DWConv,  [24, 3, 1]]          # 3        depthwise no-stride
  - [-1, 1, Conv,    [24, 1, 1]]          # 4        pointwise project
  - [-1, 1, DWConv,  [24, 3, 2]]          # 5-P3/8   stride=2
  - [-1, 1, Conv,    [48, 1, 1]]          # 6        expand
  - [-1, 1, C2f,     [48, False]]         # 7        P3 feature
  - [-1, 1, DWConv,  [48, 3, 2]]          # 8-P4/16  stride=2
  - [-1, 1, Conv,    [96, 1, 1]]          # 9        expand
  - [-1, 1, C2f,     [96, False]]         # 10       P4 feature
  - [-1, 1, DWConv,  [96, 3, 2]]          # 11-P5/32 stride=2
  - [-1, 1, Conv,    [160, 1, 1]]         # 12       expand
  - [-1, 1, SPPF,    [160, 5]]            # 13       P5 feature

head:
  - [-1,    1, nn.Upsample, [None, 2, 'nearest']]  # 14
  - [[-1,10], 1, Concat, [1]]                       # 15  160+96=256
  - [-1,    1, C2f,  [96, False]]                   # 16
  - [-1,    1, nn.Upsample, [None, 2, 'nearest']]  # 17
  - [[-1, 7], 1, Concat, [1]]                       # 18  96+48=144
  - [-1,    1, C2f,  [48, False]]                   # 19  P3 out
  - [-1,    1, DWConv, [48, 3, 2]]                  # 20
  - [[-1,16], 1, Concat, [1]]                       # 21  48+96=144
  - [-1,    1, C2f,  [96, False]]                   # 22  P4 out
  - [-1,    1, DWConv, [96, 3, 2]]                  # 23
  - [[-1,13], 1, Concat, [1]]                       # 24  96+160=256
  - [-1,    1, C2f,  [160, False]]                  # 25  P5 out
  - [[19, 22, 25], 1, Detect, [nc]]                 # 26
"""

with open(MOBILENET_FULL_YAML, 'w') as f:
    f.write(mobilenet_full)
print(f"✓ {MOBILENET_FULL_YAML.name}")

In [ ]:
# ── 2. MobileNetV2 Reduced  (8→16→32→64→96  — all channels halved) ───────────
MOBILENET_REDUCED_YAML = YAML_DIR / "yolov8_mobilenet_reduced.yaml"

mobilenet_reduced = """\
# MobileNetV2-style backbone (REDUCED: 8->16->32->64->96 channels) ~0.9M params
nc: 2
scales:
  n: [0.33, 0.25, 1024]

backbone:
  - [-1, 1, Conv,    [8,  3, 2]]          # 0-P1/2
  - [-1, 1, DWConv,  [8,  3, 2]]          # 1-P2/4  depthwise
  - [-1, 1, Conv,    [16, 1, 1]]          # 2        pointwise expand
  - [-1, 1, DWConv,  [16, 3, 1]]          # 3        depthwise no-stride
  - [-1, 1, Conv,    [16, 1, 1]]          # 4        pointwise project
  - [-1, 1, DWConv,  [16, 3, 2]]          # 5-P3/8   stride=2
  - [-1, 1, Conv,    [32, 1, 1]]          # 6        expand
  - [-1, 1, C2f,     [32, False]]         # 7        P3 feature
  - [-1, 1, DWConv,  [32, 3, 2]]          # 8-P4/16  stride=2
  - [-1, 1, Conv,    [64, 1, 1]]          # 9        expand
  - [-1, 1, C2f,     [64, False]]         # 10       P4 feature
  - [-1, 1, DWConv,  [64, 3, 2]]          # 11-P5/32 stride=2
  - [-1, 1, Conv,    [96, 1, 1]]          # 12       expand
  - [-1, 1, SPPF,    [96, 5]]             # 13       P5 feature

head:
  - [-1,    1, nn.Upsample, [None, 2, 'nearest']]  # 14
  - [[-1,10], 1, Concat, [1]]                       # 15  96+64=160
  - [-1,    1, C2f,  [64, False]]                   # 16
  - [-1,    1, nn.Upsample, [None, 2, 'nearest']]  # 17
  - [[-1, 7], 1, Concat, [1]]                       # 18  64+32=96
  - [-1,    1, C2f,  [32, False]]                   # 19  P3 out
  - [-1,    1, DWConv, [32, 3, 2]]                  # 20
  - [[-1,16], 1, Concat, [1]]                       # 21  32+64=96
  - [-1,    1, C2f,  [64, False]]                   # 22  P4 out
  - [-1,    1, DWConv, [64, 3, 2]]                  # 23
  - [[-1,13], 1, Concat, [1]]                       # 24  64+96=160
  - [-1,    1, C2f,  [96, False]]                   # 25  P5 out
  - [[19, 22, 25], 1, Detect, [nc]]                 # 26
"""

with open(MOBILENET_REDUCED_YAML, 'w') as f:
    f.write(mobilenet_reduced)
print(f"✓ {MOBILENET_REDUCED_YAML.name}")

In [ ]:
# ── 3. GhostNet Full  (16→32→64→128→160) ─────────────────────────────────────
GHOST_FULL_YAML = YAML_DIR / "yolov8_ghost_full.yaml"

ghost_full = """\
# GhostNet-style backbone (FULL: 16->32->64->128->160 channels) ~1.5M params
# GhostConv: primary conv (half filters) + cheap ghost maps via depthwise
# C3Ghost:   C3 bottleneck with GhostConv inner layers
nc: 2
scales:
  n: [0.33, 0.25, 1024]

backbone:
  - [-1, 1, Conv,      [16,  3, 2]]       # 0-P1/2   standard entry conv
  - [-1, 1, GhostConv, [32,  3, 2]]       # 1-P2/4   ghost downsample
  - [-1, 1, C3Ghost,   [32,  False]]      # 2        ghost bottleneck
  - [-1, 1, GhostConv, [64,  3, 2]]       # 3-P3/8   ghost downsample
  - [-1, 1, C3Ghost,   [64,  False]]      # 4        P3 feature
  - [-1, 1, GhostConv, [128, 3, 2]]       # 5-P4/16  ghost downsample
  - [-1, 1, C3Ghost,   [128, False]]      # 6        P4 feature
  - [-1, 1, GhostConv, [160, 3, 2]]       # 7-P5/32  ghost downsample
  - [-1, 1, SPPF,      [160, 5]]          # 8        P5 feature (multi-scale pool)

head:
  - [-1,   1, nn.Upsample, [None, 2, 'nearest']]  # 9
  - [[-1,6], 1, Concat, [1]]                        # 10  160+128=288
  - [-1,   1, C3Ghost, [128, False]]                # 11
  - [-1,   1, nn.Upsample, [None, 2, 'nearest']]  # 12
  - [[-1,4], 1, Concat, [1]]                        # 13  128+64=192
  - [-1,   1, C3Ghost, [64,  False]]                # 14  P3 out
  - [-1,   1, GhostConv, [64,  3, 2]]               # 15
  - [[-1,11],1, Concat, [1]]                        # 16  64+128=192
  - [-1,   1, C3Ghost, [128, False]]                # 17  P4 out
  - [-1,   1, GhostConv, [128, 3, 2]]               # 18
  - [[-1, 8],1, Concat, [1]]                        # 19  128+160=288
  - [-1,   1, C3Ghost, [160, False]]                # 20  P5 out
  - [[14, 17, 20], 1, Detect, [nc]]                 # 21
"""

with open(GHOST_FULL_YAML, 'w') as f:
    f.write(ghost_full)
print(f"✓ {GHOST_FULL_YAML.name}")

In [ ]:
# ── 4. GhostNet Reduced  (8→16→32→64→96  — all channels halved) ──────────────
GHOST_REDUCED_YAML = YAML_DIR / "yolov8_ghost_reduced.yaml"

ghost_reduced = """\
# GhostNet-style backbone (REDUCED: 8->16->32->64->96 channels) ~0.7M params
nc: 2
scales:
  n: [0.33, 0.25, 1024]

backbone:
  - [-1, 1, Conv,      [8,  3, 2]]        # 0-P1/2   standard entry conv
  - [-1, 1, GhostConv, [16, 3, 2]]        # 1-P2/4   ghost downsample
  - [-1, 1, C3Ghost,   [16, False]]       # 2        ghost bottleneck
  - [-1, 1, GhostConv, [32, 3, 2]]        # 3-P3/8   ghost downsample
  - [-1, 1, C3Ghost,   [32, False]]       # 4        P3 feature
  - [-1, 1, GhostConv, [64, 3, 2]]        # 5-P4/16  ghost downsample
  - [-1, 1, C3Ghost,   [64, False]]       # 6        P4 feature
  - [-1, 1, GhostConv, [96, 3, 2]]        # 7-P5/32  ghost downsample
  - [-1, 1, SPPF,      [96, 5]]           # 8        P5 feature

head:
  - [-1,   1, nn.Upsample, [None, 2, 'nearest']]  # 9
  - [[-1,6], 1, Concat, [1]]                        # 10  96+64=160
  - [-1,   1, C3Ghost, [64,  False]]                # 11
  - [-1,   1, nn.Upsample, [None, 2, 'nearest']]  # 12
  - [[-1,4], 1, Concat, [1]]                        # 13  64+32=96
  - [-1,   1, C3Ghost, [32,  False]]                # 14  P3 out
  - [-1,   1, GhostConv, [32,  3, 2]]               # 15
  - [[-1,11],1, Concat, [1]]                        # 16  32+64=96
  - [-1,   1, C3Ghost, [64,  False]]                # 17  P4 out
  - [-1,   1, GhostConv, [64,  3, 2]]               # 18
  - [[-1, 8],1, Concat, [1]]                        # 19  64+96=160
  - [-1,   1, C3Ghost, [96,  False]]                # 20  P5 out
  - [[14, 17, 20], 1, Detect, [nc]]                 # 21
"""

with open(GHOST_REDUCED_YAML, 'w') as f:
    f.write(ghost_reduced)
print(f"✓ {GHOST_REDUCED_YAML.name}")

In [ ]:
# Build all 4 models and compare parameter counts
print("Building models — counting parameters...\n")

model_yamls = {
    'MobileNetV2 Full  (~1.8M)': MOBILENET_FULL_YAML,
    'MobileNetV2 Reduced (~0.9M)': MOBILENET_REDUCED_YAML,
    'GhostNet Full   (~1.5M)': GHOST_FULL_YAML,
    'GhostNet Reduced (~0.7M)': GHOST_REDUCED_YAML,
}

yolov8n = YOLO('yolov8n.pt')
yolov8n_params = sum(p.numel() for p in yolov8n.model.parameters())

print(f"{'='*62}")
print(f"{'Model':<35} {'Params':>10} {'vs YOLOv8n':>12}")
print(f"{'='*62}")
print(f"{'YOLOv8n (CSPDarknet baseline)':<35} {yolov8n_params/1e6:>8.2f}M {'100%':>12}")

built_models = {}
for name, yaml_path in model_yamls.items():
    m = YOLO(str(yaml_path))
    params = sum(p.numel() for p in m.model.parameters())
    built_models[name] = params
    print(f"{name:<35} {params/1e6:>8.2f}M {params/yolov8n_params*100:>11.0f}%")

print(f"{'='*62}")

## 3. Visualize Channel Width Comparison

In [ ]:
stages = ['P1/2', 'P2/4', 'P3/8', 'P4/16', 'P5/32']
channels = {
    'YOLOv8n\n(CSPDarknet)': [16, 32,  64, 128, 256],
    'MobileNetV2\nFull':     [16, 24,  48,  96, 160],
    'MobileNetV2\nReduced':  [ 8, 16,  32,  64,  96],
    'GhostNet\nFull':        [16, 32,  64, 128, 160],
    'GhostNet\nReduced':     [ 8, 16,  32,  64,  96],
}

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(stages))
w = 0.15
colors = ['#3498db', '#e67e22', '#f39c12', '#2ecc71', '#1abc9c']

for i, (name, ch) in enumerate(channels.items()):
    bars = ax.bar(x + i*w, ch, w, label=name, color=colors[i], alpha=0.85)
    for bar, val in zip(bars, ch):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                str(val), ha='center', fontsize=7, fontweight='bold')

ax.set_xlabel('Backbone Stage'); ax.set_ylabel('Channels')
ax.set_title('Channel Width Across All Backbone Variants')
ax.set_xticks(x + w*2); ax.set_xticklabels(stages)
ax.legend(loc='upper left', fontsize=8); ax.set_ylim(0, 290)
plt.tight_layout(); plt.show()

## 4. Training Function

Same augmentation settings as `aug-85.ipynb` — applied identically to all 4 models for fair comparison.

| Param | Value | Note |
|-------|-------|------|
| epochs | 100 | with patience=30 early stopping |
| batch | 16 | same as baseline |
| optimizer | SGD | same as baseline |
| lr0 | model-specific | Full=0.01, Reduced=0.008 |
| mosaic | 0.9 | heavy |
| mixup | 0.15 | same |
| copy_paste | 0.15 | same |

In [ ]:
def train_custom_backbone(yaml_path, data_yaml, run_name,
                           epochs=100, batch=16, lr0=0.01,
                           project_dir=None):
    """
    Train a custom-backbone YOLOv8 model.
    yaml_path  : path to backbone YAML (defines architecture)
    data_yaml  : path to data.yaml
    run_name   : unique name for this run
    lr0        : initial learning rate (slightly lower for reduced models)
    """
    print(f"\n{'='*62}")
    print(f" Training: {run_name}")
    print(f" YAML    : {Path(yaml_path).name}")
    print(f"{'='*62}")

    model = YOLO(str(yaml_path))
    params = sum(p.numel() for p in model.model.parameters())
    print(f" Parameters: {params:,} ({params/1e6:.2f}M)")

    cfg = dict(
        data        = str(data_yaml),
        epochs      = epochs,
        imgsz       = 640,
        batch       = batch,
        patience    = 30,

        # Optimizer
        optimizer    = 'SGD',
        lr0          = lr0,
        lrf          = 0.01,
        momentum     = 0.937,
        weight_decay = 0.0005,
        warmup_epochs    = 3,
        warmup_momentum  = 0.8,

        # Augmentation — identical to aug-85.ipynb
        hsv_h       = 0.015,
        hsv_s       = 0.7,
        hsv_v       = 0.4,
        degrees     = 15.0,
        translate   = 0.15,
        scale       = 0.6,
        shear       = 5.0,
        perspective = 0.0005,
        flipud      = 0.3,
        fliplr      = 0.5,
        mosaic      = 0.9,
        mixup       = 0.15,
        copy_paste  = 0.15,

        # Loss weights
        box = 7.5, cls = 0.5, dfl = 1.5,

        val         = True,
        plots       = True,
        save        = True,
        save_period = 10,
        project     = str(project_dir) if project_dir else 'runs/detect',
        name        = run_name,
        exist_ok    = True,
        verbose     = True,
    )

    results = model.train(**cfg)
    print(f"\n✓ Done: {run_name}  Best: {model.trainer.best}")
    return model, results

## 5. Train All 4 Models

Run each training cell independently — or run them all sequentially.

In [ ]:
# ── Model 1: MobileNetV2 Full ─────────────────────────────────────────────────
model_mob_full, _ = train_custom_backbone(
    yaml_path   = MOBILENET_FULL_YAML,
    data_yaml   = DATA_YAML,
    run_name    = 'mobilenet_full',
    lr0         = 0.01,
    project_dir = RUNS_DIR,
)

In [ ]:
# ── Model 2: MobileNetV2 Reduced ──────────────────────────────────────────────
# lr0 slightly lower — narrower channels need gentler gradient updates
model_mob_red, _ = train_custom_backbone(
    yaml_path   = MOBILENET_REDUCED_YAML,
    data_yaml   = DATA_YAML,
    run_name    = 'mobilenet_reduced',
    lr0         = 0.008,   # ← lower for reduced model
    project_dir = RUNS_DIR,
)

In [ ]:
# ── Model 3: GhostNet Full ────────────────────────────────────────────────────
model_ghost_full, _ = train_custom_backbone(
    yaml_path   = GHOST_FULL_YAML,
    data_yaml   = DATA_YAML,
    run_name    = 'ghost_full',
    lr0         = 0.01,
    project_dir = RUNS_DIR,
)

In [ ]:
# ── Model 4: GhostNet Reduced ─────────────────────────────────────────────────
model_ghost_red, _ = train_custom_backbone(
    yaml_path   = GHOST_REDUCED_YAML,
    data_yaml   = DATA_YAML,
    run_name    = 'ghost_reduced',
    lr0         = 0.008,   # ← lower for reduced model
    project_dir = RUNS_DIR,
)

## 6. Training Curves — All 4 Models

In [ ]:
def plot_training_curves(runs_dir, run_name):
    """Plot loss and metric curves from results.csv."""
    csv_path = Path(runs_dir) / run_name / 'results.csv'
    if not csv_path.exists():
        print(f"results.csv not found: {csv_path}"); return

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    epoch = df['epoch'] if 'epoch' in df.columns else df.index + 1

    train_cols  = [c for c in df.columns if c.startswith('train/')]
    val_cols    = [c for c in df.columns if c.startswith('val/')]
    metric_cols = [c for c in df.columns if c.startswith('metrics/')]
    ncols = max(len(train_cols), len(val_cols), len(metric_cols), 3)

    fig, axes = plt.subplots(3, ncols, figsize=(5*ncols, 11))
    fig.suptitle(f'Training Curves — {run_name}', fontsize=14, fontweight='bold')

    for ax, col in zip(axes[0], train_cols):
        ax.plot(epoch, df[col], '#e74c3c', lw=1.8)
        ax.set_title(col.replace('train/','Train ')); ax.grid(alpha=0.3)
    for ax in axes[0][len(train_cols):]: ax.set_visible(False)

    for ax, col in zip(axes[1], val_cols):
        ax.plot(epoch, df[col], '#3498db', lw=1.8)
        ax.set_title(col.replace('val/','Val ')); ax.grid(alpha=0.3)
    for ax in axes[1][len(val_cols):]: ax.set_visible(False)

    m_colors = ['#2ecc71','#9b59b6','#f39c12','#1abc9c']
    for ax, col, c in zip(axes[2], metric_cols, m_colors):
        ax.plot(epoch, df[col], c, lw=1.8)
        ax.set_title(col.replace('metrics/','')); ax.set_ylim(0,1); ax.grid(alpha=0.3)
    for ax in axes[2][len(metric_cols):]: ax.set_visible(False)

    plt.tight_layout()
    out = Path(runs_dir) / run_name / 'training_curves.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
    print(f"✓ Saved: {out}")

for run in ['mobilenet_full', 'mobilenet_reduced', 'ghost_full', 'ghost_reduced']:
    plot_training_curves(RUNS_DIR, run)

## 7. Test Set Evaluation — All 4 Models

In [ ]:
def evaluate_model(run_dir, run_name, data_yaml, split='test'):
    """Evaluate best.pt on the given split."""
    best_pt = Path(run_dir) / run_name / 'weights' / 'best.pt'
    if not best_pt.exists():
        print(f"  ✗ best.pt not found: {best_pt}"); return None
    model   = YOLO(str(best_pt))
    params  = sum(p.numel() for p in model.model.parameters())
    metrics = model.val(data=str(data_yaml), split=split,
                        imgsz=640, batch=8, conf=0.001, iou=0.5,
                        verbose=False)
    result = {
        'Params (M)':  round(params/1e6, 2),
        'mAP50':       round(metrics.box.map50, 4),
        'mAP50-95':    round(metrics.box.map,   4),
        'Precision':   round(metrics.box.mp,    4),
        'Recall':      round(metrics.box.mr,    4),
    }
    return result


run_names = {
    'MobileNetV2 Full':    'mobilenet_full',
    'MobileNetV2 Reduced': 'mobilenet_reduced',
    'GhostNet Full':       'ghost_full',
    'GhostNet Reduced':    'ghost_reduced',
}

all_results = {}
print("Evaluating all models on test split...\n")
for display_name, run_name in run_names.items():
    print(f"  → {display_name}")
    r = evaluate_model(RUNS_DIR, run_name, DATA_YAML, split='test')
    if r:
        all_results[display_name] = r
        for k, v in r.items():
            print(f"       {k}: {v}")
    print()

## 8. Full Comparison: MobileNetV2 vs GhostNet × Full vs Reduced

In [ ]:
if all_results:
    df = pd.DataFrame(all_results).T
    print("\n" + "="*72)
    print("  Final Comparison — MobileNetV2 vs GhostNet (Full vs Reduced)")
    print("="*72)
    print(df.round(4).to_string())

    # Color: blue=MobileNet, green=Ghost | dark=Full, light=Reduced
    bar_colors = ['#2980b9', '#85c1e9', '#1e8449', '#82e0aa']
    x = np.arange(len(df))
    w = 0.2

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle(
        'MobileNetV2 vs GhostNet — Full vs Reduced Parameters',
        fontsize=14, fontweight='bold'
    )

    # mAP
    ax = axes[0]
    ax.bar(x-w/2, df['mAP50'],    w, label='mAP@50',    color=bar_colors, alpha=0.9)
    ax.bar(x+w/2, df['mAP50-95'], w, label='mAP@50-95', color=bar_colors, alpha=0.5)
    ax.set_title('mAP'); ax.set_xticks(x)
    ax.set_xticklabels(df.index, rotation=20, ha='right', fontsize=9)
    ax.legend(fontsize=9); ax.set_ylim(0, 1); ax.grid(alpha=0.3)

    # Precision / Recall
    ax = axes[1]
    ax.bar(x-w/2, df['Precision'], w, label='Precision', color=bar_colors, alpha=0.9)
    ax.bar(x+w/2, df['Recall'],    w, label='Recall',    color=bar_colors, alpha=0.5)
    ax.set_title('Precision / Recall'); ax.set_xticks(x)
    ax.set_xticklabels(df.index, rotation=20, ha='right', fontsize=9)
    ax.legend(fontsize=9); ax.set_ylim(0, 1); ax.grid(alpha=0.3)

    # Parameter count
    ax = axes[2]
    bars = ax.bar(x, df['Params (M)'], color=bar_colors, alpha=0.9)
    ax.set_title('Parameter Count (M)'); ax.set_xticks(x)
    ax.set_xticklabels(df.index, rotation=20, ha='right', fontsize=9)
    ax.set_ylabel('Params (M)'); ax.grid(alpha=0.3)
    for bar, v in zip(bars, df['Params (M)']):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.01,
                f'{v:.2f}M', ha='center', fontsize=9, fontweight='bold')

    # Legend patches
    legend_patches = [
        mpatches.Patch(color='#2980b9', label='MobileNetV2 Full'),
        mpatches.Patch(color='#85c1e9', label='MobileNetV2 Reduced'),
        mpatches.Patch(color='#1e8449', label='GhostNet Full'),
        mpatches.Patch(color='#82e0aa', label='GhostNet Reduced'),
    ]
    fig.legend(handles=legend_patches, loc='lower center', ncol=4,
               fontsize=10, bbox_to_anchor=(0.5, -0.04))

    plt.tight_layout()
    out = RUNS_DIR / 'backbone_comparison.png'
    RUNS_DIR.mkdir(parents=True, exist_ok=True)
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n✓ Comparison saved: {out}")

## 9. Confusion Matrix — All 4 Models

In [ ]:
def plot_confusion_matrix(run_dir, run_name, data_yaml, class_names,
                           split='test', conf_thresh=0.25, iou_thresh=0.5):
    best_pt = Path(run_dir) / run_name / 'weights' / 'best.pt'
    if not best_pt.exists():
        print(f"  ✗ Not found: {best_pt}"); return
    model = YOLO(str(best_pt))
    with open(data_yaml) as f:
        cfg = yaml.safe_load(f)
    dataset_root = Path(cfg['path'])
    imgs_dir = dataset_root / cfg.get(split, f'{split}/images')
    lbls_dir = dataset_root / split / 'labels'
    if not imgs_dir.exists():
        print(f"  ✗ Images dir missing: {imgs_dir}"); return

    nc = len(class_names)
    cm = np.zeros((nc+1, nc+1), dtype=int)
    img_files = sorted(list(imgs_dir.glob('*.png')) + list(imgs_dir.glob('*.jpg')))

    for img_path in img_files:
        lbl_path = lbls_dir / f"{img_path.stem}.txt"
        img = cv2.imread(str(img_path))
        if img is None: continue
        h, w = img.shape[:2]
        gt_boxes = []
        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f:
                    p = line.strip().split()
                    if len(p) < 5: continue
                    cls = int(p[0])
                    xc,yc,bw,bh = map(float,p[1:5])
                    gt_boxes.append([cls,(xc-bw/2)*w,(yc-bh/2)*h,(xc+bw/2)*w,(yc+bh/2)*h])
        preds = model.predict(str(img_path), conf=conf_thresh, verbose=False)[0]
        pred_boxes = []
        if preds.boxes is not None:
            for box,conff,cls in zip(preds.boxes.xyxy.cpu().numpy(),
                                     preds.boxes.conf.cpu().numpy(),
                                     preds.boxes.cls.cpu().numpy().astype(int)):
                pred_boxes.append([int(cls),*box])
        matched_gt, matched_pred = set(), set()
        for pi,(pcls,px1,py1,px2,py2) in enumerate(pred_boxes):
            best_iou, best_gi = 0.0, -1
            for gi,(gcls,gx1,gy1,gx2,gy2) in enumerate(gt_boxes):
                if gi in matched_gt: continue
                xi1,yi1=max(px1,gx1),max(py1,gy1); xi2,yi2=min(px2,gx2),min(py2,gy2)
                inter=max(0,xi2-xi1)*max(0,yi2-yi1)
                iou=inter/((px2-px1)*(py2-py1)+(gx2-gx1)*(gy2-gy1)-inter+1e-9)
                if iou>best_iou: best_iou,best_gi=iou,gi
            if best_iou>=iou_thresh:
                cm[gt_boxes[best_gi][0]][pcls]+=1; matched_gt.add(best_gi)
            else:
                cm[nc][pcls]+=1
        for gi,(gcls,*_) in enumerate(gt_boxes):
            if gi not in matched_gt: cm[gcls][nc]+=1

    labels = class_names + ['background']
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f'Confusion Matrix — {run_name}', fontsize=13, fontweight='bold')
    for ax, data, title, fmt in zip(axes, [cm, cm_norm],
                                    ['Raw counts','Normalised'],['d','.2f']):
        sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                    xticklabels=labels, yticklabels=labels,
                    linewidths=0.5, ax=ax)
        ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title(title)
    plt.tight_layout()
    out = Path(run_dir)/run_name/f'confusion_matrix_{split}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
    print(f"✓ {run_name}: {out.name}")


for run_name in ['mobilenet_full', 'mobilenet_reduced', 'ghost_full', 'ghost_reduced']:
    plot_confusion_matrix(RUNS_DIR, run_name, DATA_YAML, CLASS_NAMES)

## 10. PR Curves — All 4 Models

In [ ]:
def compute_iou(b1, b2):
    xi1,yi1=max(b1[0],b2[0]),max(b1[1],b2[1])
    xi2,yi2=min(b1[2],b2[2]),min(b1[3],b2[3])
    inter=max(0,xi2-xi1)*max(0,yi2-yi1)
    return inter/((b1[2]-b1[0])*(b1[3]-b1[1])+(b2[2]-b2[0])*(b2[3]-b2[1])-inter+1e-9)


def compute_pr_curve(run_dir, run_name, data_yaml, class_names,
                     split='test', iou_thresh=0.5, n_conf=300):
    best_pt = Path(run_dir)/run_name/'weights'/'best.pt'
    if not best_pt.exists(): return None, None
    model = YOLO(str(best_pt))
    with open(data_yaml) as f:
        cfg = yaml.safe_load(f)
    imgs_dir = Path(cfg['path'])/cfg.get(split,f'{split}/images')
    lbls_dir = Path(cfg['path'])/split/'labels'

    nc = len(class_names)
    per_class_dets = defaultdict(list)
    gt_counts = Counter()

    for img_path in sorted(list(imgs_dir.glob('*.png'))+list(imgs_dir.glob('*.jpg'))):
        lbl_path = lbls_dir/f"{img_path.stem}.txt"
        img = cv2.imread(str(img_path))
        if img is None: continue
        h,w = img.shape[:2]
        gt_by_cls = defaultdict(list)
        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f:
                    p=line.strip().split()
                    if len(p)<5: continue
                    cls=int(p[0]); xc,yc,bw,bh=map(float,p[1:5])
                    gt_by_cls[cls].append([(xc-bw/2)*w,(yc-bh/2)*h,(xc+bw/2)*w,(yc+bh/2)*h])
                    gt_counts[cls]+=1
        preds = model.predict(str(img_path),conf=0.001,verbose=False)[0]
        matched = defaultdict(set)
        if preds.boxes is not None:
            for box,conf,cls in sorted(
                zip(preds.boxes.xyxy.cpu().numpy(),
                    preds.boxes.conf.cpu().numpy(),
                    preds.boxes.cls.cpu().numpy().astype(int)),key=lambda x:-x[1]):
                is_tp=False
                for gi,gt_box in enumerate(gt_by_cls.get(cls,[])):
                    if gi in matched[cls]: continue
                    if compute_iou(box,gt_box)>=iou_thresh:
                        is_tp=True; matched[cls].add(gi); break
                per_class_dets[cls].append((float(conf),is_tp))

    thresholds = np.linspace(0,1,n_conf)
    ap_per_cls = {}
    pr_curves  = {}
    for cls in range(nc):
        dets = sorted(per_class_dets.get(cls,[]),key=lambda x:-x[0])
        n_gt = gt_counts.get(cls,0)
        if not dets or n_gt==0:
            pr_curves[cls]=([0,1],[1,0]); ap_per_cls[cls]=0.0; continue
        confs=np.array([d[0] for d in dets]); tps=np.array([d[1] for d in dets],float)
        recs,precs=[],[]
        for t in thresholds:
            mask=confs>=t; tp_s=tps[mask].sum(); n_p=mask.sum()
            precs.append(tp_s/(n_p+1e-9) if n_p>0 else 1.0)
            recs.append(tp_s/(n_gt+1e-9))
        recs=np.array(recs); precs=np.array(precs)
        order=np.argsort(recs); recs,precs=recs[order],precs[order]
        recs=np.concatenate([[0],recs,[recs[-1]]])
        precs=np.concatenate([[1],precs,[0]])
        for i in range(len(precs)-2,-1,-1): precs[i]=max(precs[i],precs[i+1])
        ap=float(np.trapezoid(precs,recs))
        pr_curves[cls]=(recs,precs); ap_per_cls[cls]=round(ap,4)

    mAP50 = round(float(np.mean(list(ap_per_cls.values()))),4)
    return pr_curves, ap_per_cls, mAP50, gt_counts


# Overlay PR curves from all 4 models on one plot
run_display = {
    'mobilenet_full':    ('MobileNetV2 Full',    '#2980b9', '-'),
    'mobilenet_reduced': ('MobileNetV2 Reduced', '#85c1e9', '--'),
    'ghost_full':        ('GhostNet Full',        '#1e8449', '-'),
    'ghost_reduced':     ('GhostNet Reduced',     '#82e0aa', '--'),
}

fig, axes = plt.subplots(1, NC, figsize=(8*NC, 6))
fig.suptitle('PR Curves @ IoU=0.5 — All Backbones', fontsize=14, fontweight='bold')

for run_name, (display_name, color, ls) in run_display.items():
    result = compute_pr_curve(RUNS_DIR, run_name, DATA_YAML, CLASS_NAMES)
    if result[0] is None: continue
    pr_curves, ap_per_cls, mAP50, _ = result
    for cls in range(NC):
        recs, precs = pr_curves[cls]
        cls_name = CLASS_NAMES[cls]
        axes[cls].plot(recs, precs, color=color, ls=ls, lw=2.2,
                       label=f"{display_name}  AP={ap_per_cls[cls]:.3f}")

for cls in range(NC):
    axes[cls].set_title(f'Class: {CLASS_NAMES[cls]}', fontsize=12, fontweight='bold')
    axes[cls].set_xlabel('Recall'); axes[cls].set_ylabel('Precision')
    axes[cls].set_xlim(0,1); axes[cls].set_ylim(0,1.05)
    axes[cls].legend(fontsize=9, loc='lower left'); axes[cls].grid(alpha=0.3)

plt.tight_layout()
out = RUNS_DIR / 'pr_curves_all_backbones.png'
RUNS_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f"✓ PR curves saved: {out}")

## 11. Export Best Models

In [ ]:
export_runs = {
    'mobilenet_full':    'best_mobilenet_full.pt',
    'mobilenet_reduced': 'best_mobilenet_reduced.pt',
    'ghost_full':        'best_ghost_full.pt',
    'ghost_reduced':     'best_ghost_reduced.pt',
}

print("Exporting best weights...\n")
for run_name, output_name in export_runs.items():
    best_pt = RUNS_DIR / run_name / 'weights' / 'best.pt'
    if best_pt.exists():
        dst = BASE_DIR / output_name
        shutil.copy2(best_pt, dst)
        model  = YOLO(str(dst))
        params = sum(p.numel() for p in model.model.parameters())
        onnx   = model.export(format='onnx', imgsz=640, simplify=True,
                               dynamic=False, opset=12)
        print(f"  ✓ {run_name:<22} → {output_name}  ({params/1e6:.2f}M)  ONNX: {Path(onnx).name}")
    else:
        print(f"  ✗ {run_name}: best.pt not found — run training first")

## Summary

### Architecture Summary

| Model | Backbone Key Block | Channels | Params | Neck/Head |
|-------|--------------------|----------|--------|----------|
| **MobileNetV2 Full** | DWConv 3×3 + Conv 1×1 | 16→24→48→96→160 | ~1.8M | PANet + Detect |
| **MobileNetV2 Reduced** | DWConv 3×3 + Conv 1×1 | 8→16→32→64→96 | ~0.9M | PANet + Detect |
| **GhostNet Full** | GhostConv + C3Ghost | 16→32→64→128→160 | ~1.5M | PANet + Detect |
| **GhostNet Reduced** | GhostConv + C3Ghost | 8→16→32→64→96 | ~0.7M | PANet + Detect |

### How Parameters Were Reduced

Each stage's channel width was **halved** (16→8, 24→16, 48→32, 96→64, 160→96).  
The neck uses the same narrower widths, compounding the reduction.  
`C2f`/`C3Ghost` ratios unchanged — only width changes.

### Learning Rate Adjustment
- **Full models**: `lr0=0.01` (same as `aug-85.ipynb` baseline)
- **Reduced models**: `lr0=0.008` — narrower layers have smaller gradient magnitudes, gentle updates help stability

### When to Use Which Model

| Scenario | Recommended |
|----------|------------|
| Highest accuracy, full server hardware | GhostNet Full or MobileNetV2 Full |
| Edge device / limited VRAM | GhostNet Reduced (~0.7M) |
| Best accuracy-per-parameter tradeoff | GhostNet Full (~1.5M, fewer ops than MobileNetV2) |
| Minimum size for deployment | MobileNetV2 Reduced (~0.9M, well-tested architecture) |

### Output Files
```
runs/backbone_comparison/
  mobilenet_full/weights/best.pt
  mobilenet_reduced/weights/best.pt
  ghost_full/weights/best.pt
  ghost_reduced/weights/best.pt
  backbone_comparison.png
  pr_curves_all_backbones.png
best_mobilenet_full.pt
best_mobilenet_reduced.pt
best_ghost_full.pt
best_ghost_reduced.pt
```